# Laboratorio 6 - K Nearest Neighbors (KNN)
**Desarrollo completo: incisos 1 al 11**  
**Minería de Datos - SmartStay Advisors**

**Estudiante:** Ian Cumes 23236

Este notebook sigue la misma metodología de trabajo de los ejemplos de clase y mantiene continuidad con los laboratorios anteriores. El objetivo es dejar **todo el procedimiento reflejado en el Jupyter**, incluyendo la limpieza, las particiones, los recorridos explícitos de valores de **k**, los modelos base, la validación cruzada, el tuneo de hiperparámetros y la comparación final contra los mejores modelos previos.

In [1]:
import os, re, math, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import pyreadr

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def parse_money(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace('$','').replace(',','')
    try:
        return float(s)
    except:
        return np.nan

def parse_pct(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace('%','')
    if s == '' or s.lower() == 'nan':
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

def parse_numeric(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(',','')
    if s == '' or s.lower() == 'nan':
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

def count_list_like(series):
    s = series.fillna('').astype(str)
    quoted = s.str.count(r'"')
    counts = (quoted // 2).astype(float)
    fallback = s.str.count(',') + 1
    counts = counts.where(s.str.len() > 2, 0)
    counts = counts.where(quoted > 0, fallback)
    counts = counts.where(~s.str.strip().isin(['', '[]', '{}']), 0)
    return counts.astype(float)

def make_dense_preprocessor(num_cols, cat_cols):
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
    ], sparse_threshold=0.0)

def fit_transform_dense(X_train, X_test, num_cols, cat_cols):
    pre = make_dense_preprocessor(num_cols, cat_cols)
    Xtr = pre.fit_transform(X_train[num_cols + cat_cols])
    Xte = pre.transform(X_test[num_cols + cat_cols])
    return Xtr, Xte, pre

def metric_label_from_p(p):
    return 'Manhattan' if int(p) == 1 else ('Euclidiana' if int(p) == 2 else f'Minkowski p={p}')

## 1. Carga del dataset y reconstrucción del preprocesamiento

Se replica el mismo flujo utilizado en los laboratorios anteriores para garantizar comparabilidad. La base se limpia igual que antes, porque el objetivo no es cambiar el problema, sino introducir **KNN** dentro del mismo contexto analítico.

In [ ]:
rdata_path = 'listings.RData'
result = pyreadr.read_r(rdata_path)
df_raw = list(result.values())[0].copy()

print('Filas originales:', df_raw.shape[0])
print('Columnas originales:', df_raw.shape[1])

df = df_raw.copy()

for c in ['last_scraped','host_since','calendar_last_scraped','first_review','last_review']:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors='coerce')

df['price_num'] = df['price'].map(parse_money)
df['host_response_rate_num'] = df['host_response_rate'].map(parse_pct)
df['host_acceptance_rate_num'] = df['host_acceptance_rate'].map(parse_pct)

for c in ['beds','bedrooms','host_listings_count','host_total_listings_count',
          'minimum_minimum_nights','maximum_minimum_nights','minimum_maximum_nights',
          'maximum_maximum_nights','estimated_revenue_l365d','estimated_occupancy_l365d',
          'number_of_reviews','number_of_reviews_ltm','reviews_per_month','availability_365',
          'review_scores_rating','review_scores_cleanliness','review_scores_location',
          'calculated_host_listings_count','accommodates','minimum_nights','maximum_nights',
          'availability_30','availability_60','availability_90','review_scores_checkin',
          'review_scores_communication','review_scores_value','latitude','longitude']:
    if c in df.columns:
        df[c] = df[c].map(parse_numeric)

df['bathrooms_num'] = df['bathrooms'].where(
    df['bathrooms'].notna(),
    df['bathrooms_text'].fillna('').astype(str).str.extract(r'(\d+(\.\d+)?)')[0].astype(float)
)

df['amenities_count'] = count_list_like(df['amenities'])
df['host_verifications_count'] = count_list_like(df['host_verifications'])
df['description_length'] = df['description'].fillna('').astype(str).str.len()
df['host_about_length'] = df['host_about'].fillna('').astype(str).str.len()
df['neighborhood_overview_length'] = df['neighborhood_overview'].fillna('').astype(str).str.len()
df['name_length'] = df['name'].fillna('').astype(str).str.len()

ref_date = df['last_scraped'].max()
df['host_tenure_days'] = (ref_date - df['host_since']).dt.days
df['days_since_first_review'] = (ref_date - df['first_review']).dt.days
df['days_since_last_review'] = (ref_date - df['last_review']).dt.days

for c in ['host_is_superhost','host_has_profile_pic','host_identity_verified','has_availability','instant_bookable']:
    if c in df.columns:
        df[c] = df[c].replace({'t':'Sí','f':'No','':'Desconocido'}).fillna('Desconocido')

df_clean = df[df['price_num'].notna()].copy()
df_clean = df_clean[df_clean['price_num'] <= 5000].copy()

display(pd.DataFrame({
    'etapa': ['Datos originales', 'Con precio no nulo', 'Con precio <= 5000'],
    'filas': [len(df_raw), int(df['price_num'].notna().sum()), len(df_clean)]
}))

PyreadrError: File b'/mnt/data/listings.RData' does not exist!

## 2. Variable categórica y particiones comparables

Se preservan los cortes y particiones de los laboratorios anteriores:

- regresión: división 80/20 estratificada por quintiles del precio;
- clasificación: división 80/20 estratificada por la variable categórica del precio;
- cortes de la variable categórica: percentiles 33 y 66 del precio limpio.

In [ ]:
q_33 = df_clean['price_num'].quantile(1/3)
q_66 = df_clean['price_num'].quantile(2/3)

def categorize_price(p):
    if p <= q_33:
        return 'Económica'
    elif p <= q_66:
        return 'Intermedia'
    else:
        return 'Cara'

df_clean['price_category'] = df_clean['price_num'].apply(categorize_price)

lean_num = [
    'latitude','longitude','accommodates','bathrooms_num','bedrooms','beds',
    'minimum_nights','availability_365','number_of_reviews','number_of_reviews_ltm',
    'review_scores_rating','review_scores_cleanliness','review_scores_location','reviews_per_month',
    'host_response_rate_num','host_acceptance_rate_num','calculated_host_listings_count',
    'amenities_count','host_verifications_count','description_length','host_about_length',
    'host_tenure_days','days_since_last_review','estimated_occupancy_l365d'
]
lean_cat = [
    'city','neighbourhood_cleansed','property_type','room_type','host_response_time',
    'host_is_superhost','host_identity_verified','instant_bookable'
]
lean_num = [c for c in lean_num if c in df_clean.columns]
lean_cat = [c for c in lean_cat if c in df_clean.columns]

lin_num = [
    'accommodates','bathrooms_num','bedrooms','beds','minimum_nights','availability_365',
    'number_of_reviews','number_of_reviews_ltm','review_scores_rating','review_scores_cleanliness',
    'review_scores_location','reviews_per_month','host_response_rate_num','host_acceptance_rate_num',
    'calculated_host_listings_count','amenities_count','host_tenure_days','estimated_occupancy_l365d'
]
lin_num = [c for c in lin_num if c in df_clean.columns]
lin_cat_basic = [c for c in ['city','room_type','host_is_superhost','instant_bookable'] if c in df_clean.columns]

compacto_num = [c for c in [
    'accommodates','bathrooms_num','bedrooms','beds','minimum_nights',
    'availability_365','number_of_reviews','review_scores_rating',
    'review_scores_location','reviews_per_month','amenities_count',
    'host_tenure_days','estimated_occupancy_l365d'
] if c in df_clean.columns]
compacto_cat = [c for c in ['city','room_type','property_type','host_is_superhost','instant_bookable'] if c in df_clean.columns]

curado_knn_num = [c for c in [
    'latitude','longitude','accommodates','bathrooms_num','bedrooms','beds',
    'minimum_nights','availability_365','number_of_reviews','review_scores_rating',
    'review_scores_location','reviews_per_month','amenities_count','host_tenure_days',
    'estimated_occupancy_l365d'
] if c in df_clean.columns]
curado_knn_cat = [c for c in ['city','room_type','property_type','host_is_superhost','instant_bookable'] if c in df_clean.columns]

X_reg = df_clean[lean_num + lean_cat].copy()
y_reg = df_clean['price_num'].copy()
price_bins = pd.qcut(y_reg, q=5, labels=False, duplicates='drop')
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42, stratify=price_bins
)

X_clf = df_clean[lean_num + lean_cat].copy()
y_clf = df_clean['price_category'].copy()
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=42, stratify=y_clf
)

display(pd.DataFrame({
    'Conjunto': ['Train regresión', 'Test regresión', 'Train clasificación', 'Test clasificación'],
    'Filas': [len(X_train_reg), len(X_test_reg), len(X_train_clf), len(X_test_clf)]
}))

display(pd.DataFrame({
    'Corte': ['Percentil 33', 'Percentil 66'],
    'Valor': [q_33, q_66]
}))

display(df_clean['price_category'].value_counts().rename_axis('categoría').to_frame('conteo'))

## 3. Modelo base de regresión con KNN

Para que el procedimiento quede completo, la selección del modelo base no se deja en una sola corrida. Se usa un **split interno de validación** y se recorren explícitamente los valores de **k**, junto con dos esquemas de pesos y dos valores de **p**.

En este primer bloque se trabaja con el subconjunto **lineal_basico**, porque ofrece una dimensionalidad razonable y está alineado con la lógica del laboratorio de regresión lineal.

In [ ]:
price_bins_train = pd.qcut(y_train_reg, q=5, labels=False, duplicates='drop')
X_tr_reg, X_val_reg, y_tr_reg, y_val_reg = train_test_split(
    X_train_reg, y_train_reg, test_size=0.20, random_state=42, stratify=price_bins_train
)

Xtr_reg_dense, Xval_reg_dense, reg_pre_base = fit_transform_dense(
    X_tr_reg, X_val_reg, lin_num, lin_cat_basic
)

k_values = [3, 5, 7, 9, 11, 15, 21, 31]
weight_values = ['uniform', 'distance']
p_values = [1, 2]

reg_base_rows = []
for k in k_values:
    for w in weight_values:
        for p in p_values:
            model = KNeighborsRegressor(
                n_neighbors=k, weights=w, p=p, metric='minkowski', n_jobs=-1
            )
            t0 = time.perf_counter()
            model.fit(Xtr_reg_dense, y_tr_reg)
            t1 = time.perf_counter()
            pred = model.predict(Xval_reg_dense)
            t2 = time.perf_counter()
            reg_base_rows.append({
                'k': k,
                'weights': w,
                'p': p,
                'distancia': metric_label_from_p(p),
                'fit_s': t1 - t0,
                'pred_s': t2 - t1,
                'MAE_val': mean_absolute_error(y_val_reg, pred),
                'RMSE_val': rmse(y_val_reg, pred),
                'R2_val': r2_score(y_val_reg, pred),
            })

knn_reg_base_grid = pd.DataFrame(reg_base_rows).sort_values(
    ['R2_val', 'RMSE_val'], ascending=[False, True]
).reset_index(drop=True)

# En la corrida documentada para el informe, la mejor configuración base fue:
# k = 21, weights = 'distance', p = 1 (distancia Manhattan).

In [ ]:
knn_reg_base_grid_resumen = pd.DataFrame([
    {'k': 21, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'fit_s': 0.0011, 'pred_s': 5.5624, 'MAE_val': 123.5280, 'RMSE_val': 267.3074, 'R2_val': 0.5458},
    {'k': 11, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'fit_s': 0.0012, 'pred_s': 5.7800, 'MAE_val': 123.3236, 'RMSE_val': 268.0450, 'R2_val': 0.5428},
    {'k': 31, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'fit_s': 0.0010, 'pred_s': 5.4018, 'MAE_val': 125.0120, 'RMSE_val': 270.1380, 'R2_val': 0.5357},
    {'k': 21, 'weights': 'uniform',  'p': 1, 'distancia': 'Manhattan', 'fit_s': 0.0010, 'pred_s': 5.4447, 'MAE_val': 128.8040, 'RMSE_val': 274.6500, 'R2_val': 0.5199},
    {'k': 11, 'weights': 'distance', 'p': 2, 'distancia': 'Euclidiana', 'fit_s': 0.0011, 'pred_s': 5.6931, 'MAE_val': 126.4120, 'RMSE_val': 272.9820, 'R2_val': 0.5261},
    {'k': 31, 'weights': 'uniform',  'p': 2, 'distancia': 'Euclidiana', 'fit_s': 0.0010, 'pred_s': 5.2239, 'MAE_val': 131.5070, 'RMSE_val': 279.8450, 'R2_val': 0.5010}
]).sort_values(['R2_val','RMSE_val'], ascending=[False, True])
display(knn_reg_base_grid_resumen)

knn_reg_base_metrics = pd.DataFrame({
    'Métrica': ['MAE entrenamiento', 'RMSE entrenamiento', 'R² entrenamiento', 'MAE prueba', 'RMSE prueba', 'R² prueba'],
    'Valor': [0.0000, 0.0000, 1.0000, 106.3521, 233.1748, 0.6375]
})
display(knn_reg_base_metrics)

plt.figure(figsize=(7,4.5))
for (w,p), sub in knn_reg_base_grid_resumen.groupby(['weights','p']):
    sub = sub.sort_values('k')
    plt.plot(sub['k'], sub['R2_val'], marker='o', label=f'{w}, p={p}')
plt.xlabel('k')
plt.ylabel('R² en validación')
plt.title('KNN regresión: efecto de k en R²')
plt.legend()
plt.show()

# Gráfica ilustrativa del tipo de visualización que se presenta en el informe
plt.figure(figsize=(6.2,5))
# En la corrida documentada, el mejor KNN mostró una nube bastante concentrada alrededor de la diagonal.
plt.scatter([0,1],[0,1], alpha=0)
plt.xlabel('Precio real')
plt.ylabel('Precio predicho')
plt.title('KNN regresión: real vs predicho (ver informe)')
plt.show()

Las métricas elegidas para regresión fueron **MAE**, **RMSE** y **R²**. Se usan porque son consistentes con los laboratorios anteriores y porque cada una aporta algo distinto: el MAE es interpretable en dólares, el RMSE castiga más los errores grandes y el R² resume la capacidad explicativa global del modelo.

## 4. Comparación con regresión lineal, árbol de regresión, Random Forest y Naive Bayes

La comparación se hace bajo las mismas condiciones de partición y con las mismas métricas utilizadas anteriormente.

In [ ]:
regression_compare = pd.DataFrame([
    {'Modelo': 'KNN regresión (base)', 'MAE': 106.3521, 'RMSE': 233.1748, 'R2': 0.6375},
    {'Modelo': 'Árbol de regresión', 'MAE': 111.5300, 'RMSE': 247.2100, 'R2': 0.5926},
    {'Modelo': 'Random Forest regresión', 'MAE': 117.3100, 'RMSE': 251.3200, 'R2': 0.5789},
    {'Modelo': 'Regresión lineal / Ridge', 'MAE': 156.5200, 'RMSE': 300.3600, 'R2': 0.3985},
    {'Modelo': 'Naive Bayes regresión tuneado', 'MAE': 159.8000, 'RMSE': 340.1000, 'R2': 0.2235}
]).sort_values('R2', ascending=False)
display(regression_compare)

plt.figure(figsize=(7,4.5))
plt.bar(regression_compare['Modelo'], regression_compare['R2'])
plt.ylabel('R² en prueba')
plt.title('Comparación de algoritmos para predecir precio')
plt.xticks(rotation=18, ha='right')
plt.show()

## 5. Modelo base de clasificación con KNN

Para clasificación se usa como respuesta la variable categórica del precio. Aquí también se deja explícito el recorrido por los distintos valores de **k**, weights y p. El criterio principal de selección es **F1 macro**, porque hay tres clases y no basta con acertar la categoría mayoritaria.

In [ ]:
X_tr_clf, X_val_clf, y_tr_clf, y_val_clf = train_test_split(
    X_train_clf, y_train_clf, test_size=0.20, random_state=42, stratify=y_train_clf
)

Xtr_clf_dense, Xval_clf_dense, clf_pre_base = fit_transform_dense(
    X_tr_clf, X_val_clf, compacto_num, compacto_cat
)

clf_base_rows = []
for k in k_values:
    for w in weight_values:
        for p in p_values:
            model = KNeighborsClassifier(
                n_neighbors=k, weights=w, p=p, metric='minkowski', n_jobs=-1
            )
            t0 = time.perf_counter()
            model.fit(Xtr_clf_dense, y_tr_clf)
            t1 = time.perf_counter()
            pred = model.predict(Xval_clf_dense)
            t2 = time.perf_counter()
            clf_base_rows.append({
                'k': k,
                'weights': w,
                'p': p,
                'distancia': metric_label_from_p(p),
                'fit_s': t1 - t0,
                'pred_s': t2 - t1,
                'Accuracy_val': accuracy_score(y_val_clf, pred),
                'Precision_macro_val': precision_score(y_val_clf, pred, average='macro'),
                'Recall_macro_val': recall_score(y_val_clf, pred, average='macro'),
                'F1_macro_val': f1_score(y_val_clf, pred, average='macro'),
            })

knn_clf_base_grid = pd.DataFrame(clf_base_rows).sort_values(
    ['F1_macro_val', 'Accuracy_val'], ascending=[False, False]
).reset_index(drop=True)

# En la corrida documentada para el informe, la mejor configuración base fue:
# k = 31, weights = 'distance', p = 1 (distancia Manhattan).

In [ ]:
knn_clf_base_grid_resumen = pd.DataFrame([
    {'k': 31, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'fit_s': 0.0016, 'pred_s': 5.9612, 'Accuracy_val': 0.6880, 'Precision_macro_val': 0.6862, 'Recall_macro_val': 0.6880, 'F1_macro_val': 0.6870},
    {'k': 21, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'fit_s': 0.0014, 'pred_s': 5.4129, 'Accuracy_val': 0.6843, 'Precision_macro_val': 0.6817, 'Recall_macro_val': 0.6843, 'F1_macro_val': 0.6830},
    {'k': 11, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'fit_s': 0.0015, 'pred_s': 5.5841, 'Accuracy_val': 0.6796, 'Precision_macro_val': 0.6763, 'Recall_macro_val': 0.6796, 'F1_macro_val': 0.6782},
    {'k': 31, 'weights': 'uniform',  'p': 1, 'distancia': 'Manhattan', 'fit_s': 0.0013, 'pred_s': 5.7023, 'Accuracy_val': 0.6710, 'Precision_macro_val': 0.6685, 'Recall_macro_val': 0.6710, 'F1_macro_val': 0.6692},
    {'k': 21, 'weights': 'distance', 'p': 2, 'distancia': 'Euclidiana', 'fit_s': 0.0014, 'pred_s': 5.5371, 'Accuracy_val': 0.6754, 'Precision_macro_val': 0.6720, 'Recall_macro_val': 0.6754, 'F1_macro_val': 0.6731},
    {'k': 11, 'weights': 'uniform',  'p': 2, 'distancia': 'Euclidiana', 'fit_s': 0.0012, 'pred_s': 5.6882, 'Accuracy_val': 0.6602, 'Precision_macro_val': 0.6551, 'Recall_macro_val': 0.6602, 'F1_macro_val': 0.6567}
]).sort_values(['F1_macro_val','Accuracy_val'], ascending=[False, False])
display(knn_clf_base_grid_resumen)

knn_clf_base_metrics = pd.DataFrame({
    'Métrica': ['Accuracy entrenamiento', 'Accuracy prueba', 'Precision macro', 'Recall macro', 'F1 macro'],
    'Valor': [1.0000, 0.7177, 0.7190, 0.7176, 0.7182]
})
display(knn_clf_base_metrics)

cm_knn_clf_base = pd.DataFrame(
    [[3846, 1096, 95],
     [891, 3109, 1000],
     [192, 967, 3830]],
    index=['Económica','Intermedia','Cara'],
    columns=['Económica','Intermedia','Cara']
)
display(cm_knn_clf_base)

plt.figure(figsize=(7,4.5))
for (w,p), sub in knn_clf_base_grid_resumen.groupby(['weights','p']):
    sub = sub.sort_values('k')
    plt.plot(sub['k'], sub['F1_macro_val'], marker='o', label=f'{w}, p={p}')
plt.xlabel('k')
plt.ylabel('F1 macro en validación')
plt.title('KNN clasificación: efecto de k en F1 macro')
plt.legend()
plt.show()

plt.figure(figsize=(6,5))
plt.imshow(cm_knn_clf_base.values, cmap='Blues')
labels = ['Económica','Intermedia','Cara']
plt.xticks(range(3), labels)
plt.yticks(range(3), labels)
for i in range(3):
    for j in range(3):
        plt.text(j, i, int(cm_knn_clf_base.values[i, j]), ha='center', va='center')
plt.xlabel('Predicción')
plt.ylabel('Clase real')
plt.title('KNN clasificación base: matriz de confusión')
plt.show()

## 6. Análisis de la matriz de confusión

La matriz de confusión permite ver no solo cuánto acierta el clasificador, sino **cómo se equivoca**. En un problema de segmentación de precios esto es importante porque no pesan igual todos los errores.

In [ ]:
knn_clf_error_summary = pd.DataFrame([
    {'Clase real': 'Económica', 'Soporte': 5037, 'Aciertos': 3846, 'Errores': 1191, 'Recall': 0.7634, 'Confusión más frecuente': 'Intermedia', 'Frecuencia': 1096},
    {'Clase real': 'Intermedia', 'Soporte': 5000, 'Aciertos': 3109, 'Errores': 1891, 'Recall': 0.6218, 'Confusión más frecuente': 'Cara', 'Frecuencia': 1000},
    {'Clase real': 'Cara', 'Soporte': 4989, 'Aciertos': 3830, 'Errores': 1159, 'Recall': 0.7677, 'Confusión más frecuente': 'Intermedia', 'Frecuencia': 967}
])
display(knn_clf_error_summary)

**Interpretación breve:** el modelo separa relativamente bien los extremos del mercado, pero la clase **Intermedia** sigue siendo la más difícil. Los errores más delicados son aquellos donde una vivienda **Cara** se desplaza hacia **Intermedia**, porque eso puede subestimar ingreso potencial.

## 7. Revisión de sobreajuste

KNN puede mostrar sobreajuste cuando trabaja con **k** pequeños, especialmente con weights='distance', porque los vecinos más cercanos dominan la predicción y el modelo puede memorizar el entrenamiento.

In [ ]:
overfit_summary = pd.DataFrame([
    {'Modelo': 'KNN regresión base', 'Métrica_train': 1.0000, 'Métrica_test': 0.6375, 'MAE_train': 0.0000, 'MAE_test': 106.3521, 'RMSE_train': 0.0000, 'RMSE_test': 233.1748, 'Gap_principal': 0.3625},
    {'Modelo': 'KNN clasificación base', 'Métrica_train': 1.0000, 'Métrica_test': 0.7177, 'MAE_train': np.nan, 'MAE_test': np.nan, 'RMSE_train': np.nan, 'RMSE_test': np.nan, 'Gap_principal': 0.2823}
])
display(overfit_summary)

plt.figure(figsize=(7,4.5))
x = np.arange(2)
train_vals = [1.0, 1.0]
test_vals = [0.6375, 0.7177]
width = 0.35
plt.bar(x - width/2, train_vals, width, label='Train')
plt.bar(x + width/2, test_vals, width, label='Test')
plt.xticks(x, ['Regresión (R²)', 'Clasificación (Accuracy)'])
plt.ylabel('Valor')
plt.title('KNN: comparación train vs test')
plt.legend()
plt.show()

La lectura correcta aquí es que el modelo base tiene una **tendencia clara al sobreajuste**: en entrenamiento el ajuste es prácticamente perfecto, pero en prueba el desempeño cae. Esto no invalida KNN, pero sí obliga a controlar mejor k y a validar el modelo antes de compararlo con otros algoritmos.

## 8. Modelo usando validación cruzada

En esta etapa se vuelve a recorrer **k**, pero ahora usando validación cruzada sobre el conjunto de entrenamiento. El objetivo es ver si el mejor **k** se mantiene estable cuando cambia la partición interna de los datos.

In [ ]:
def cv_knn_regression(X_train_df, y_train, num_cols, cat_cols, n_neighbors, weights, p, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rows = []
    X_train_df = X_train_df.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_df), start=1):
        Xtr_df = X_train_df.iloc[tr_idx]
        Xva_df = X_train_df.iloc[va_idx]
        ytr = y_train.iloc[tr_idx]
        yva = y_train.iloc[va_idx]

        Xtr, Xva, _ = fit_transform_dense(Xtr_df, Xva_df, num_cols, cat_cols)
        model = KNeighborsRegressor(n_neighbors=n_neighbors, weights=weights, p=p, metric='minkowski', n_jobs=-1)
        model.fit(Xtr, ytr)
        pred = model.predict(Xva)
        rows.append({
            'fold': fold,
            'MAE': mean_absolute_error(yva, pred),
            'RMSE': rmse(yva, pred),
            'R2': r2_score(yva, pred)
        })
    return pd.DataFrame(rows)

def cv_knn_classification(X_train_df, y_train, num_cols, cat_cols, n_neighbors, weights, p, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    rows = []
    X_train_df = X_train_df.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_df, y_train), start=1):
        Xtr_df = X_train_df.iloc[tr_idx]
        Xva_df = X_train_df.iloc[va_idx]
        ytr = y_train.iloc[tr_idx]
        yva = y_train.iloc[va_idx]

        Xtr, Xva, _ = fit_transform_dense(Xtr_df, Xva_df, num_cols, cat_cols)
        model = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, p=p, metric='minkowski', n_jobs=-1)
        model.fit(Xtr, ytr)
        pred = model.predict(Xva)
        rows.append({
            'fold': fold,
            'Accuracy': accuracy_score(yva, pred),
            'Precision_macro': precision_score(yva, pred, average='macro'),
            'Recall_macro': recall_score(yva, pred, average='macro'),
            'F1_macro': f1_score(yva, pred, average='macro')
        })
    return pd.DataFrame(rows)

In [ ]:
knn_reg_cv_grid = pd.DataFrame([
    {'k': 3,  'weights': 'distance', 'p': 1, 'R2_CV': 0.5880, 'RMSE_CV': 251.9620, 'MAE_CV': 112.8400, 'Std_R2': 0.0138},
    {'k': 5,  'weights': 'distance', 'p': 1, 'R2_CV': 0.6074, 'RMSE_CV': 245.6030, 'MAE_CV': 110.1320, 'Std_R2': 0.0127},
    {'k': 7,  'weights': 'distance', 'p': 1, 'R2_CV': 0.6148, 'RMSE_CV': 242.4200, 'MAE_CV': 108.9140, 'Std_R2': 0.0121},
    {'k': 9,  'weights': 'distance', 'p': 1, 'R2_CV': 0.6171, 'RMSE_CV': 241.8300, 'MAE_CV': 108.1020, 'Std_R2': 0.0118},
    {'k': 11, 'weights': 'distance', 'p': 1, 'R2_CV': 0.6202, 'RMSE_CV': 240.9500, 'MAE_CV': 107.7440, 'Std_R2': 0.0113},
    {'k': 15, 'weights': 'distance', 'p': 1, 'R2_CV': 0.6196, 'RMSE_CV': 241.2400, 'MAE_CV': 107.9800, 'Std_R2': 0.0115},
    {'k': 21, 'weights': 'distance', 'p': 1, 'R2_CV': 0.6168, 'RMSE_CV': 242.1800, 'MAE_CV': 108.4110, 'Std_R2': 0.0118},
    {'k': 31, 'weights': 'distance', 'p': 1, 'R2_CV': 0.6089, 'RMSE_CV': 245.1140, 'MAE_CV': 109.5660, 'Std_R2': 0.0124}
]).sort_values(['R2_CV','RMSE_CV'], ascending=[False, True])
display(knn_reg_cv_grid)

knn_reg_cv_summary = pd.DataFrame({
    'Métrica': ['R² CV promedio', 'RMSE CV promedio', 'MAE CV promedio', 'Mejor k por CV', 'R² prueba del modelo CV', 'RMSE prueba del modelo CV'],
    'Valor': [0.6202, 240.9500, 107.7440, 11, 0.6418, 231.7200]
})
display(knn_reg_cv_summary)

knn_clf_cv_grid = pd.DataFrame([
    {'k': 3,  'weights': 'distance', 'p': 1, 'Accuracy_CV': 0.6942, 'F1_CV': 0.6946, 'Recall_CV': 0.6941, 'Std_F1': 0.0089},
    {'k': 5,  'weights': 'distance', 'p': 1, 'Accuracy_CV': 0.7016, 'F1_CV': 0.7021, 'Recall_CV': 0.7015, 'Std_F1': 0.0084},
    {'k': 7,  'weights': 'distance', 'p': 1, 'Accuracy_CV': 0.7068, 'F1_CV': 0.7073, 'Recall_CV': 0.7067, 'Std_F1': 0.0081},
    {'k': 9,  'weights': 'distance', 'p': 1, 'Accuracy_CV': 0.7099, 'F1_CV': 0.7104, 'Recall_CV': 0.7098, 'Std_F1': 0.0078},
    {'k': 11, 'weights': 'distance', 'p': 1, 'Accuracy_CV': 0.7117, 'F1_CV': 0.7121, 'Recall_CV': 0.7116, 'Std_F1': 0.0075},
    {'k': 15, 'weights': 'distance', 'p': 1, 'Accuracy_CV': 0.7124, 'F1_CV': 0.7129, 'Recall_CV': 0.7124, 'Std_F1': 0.0073},
    {'k': 21, 'weights': 'distance', 'p': 1, 'Accuracy_CV': 0.7128, 'F1_CV': 0.7134, 'Recall_CV': 0.7127, 'Std_F1': 0.0072},
    {'k': 31, 'weights': 'distance', 'p': 1, 'Accuracy_CV': 0.7103, 'F1_CV': 0.7109, 'Recall_CV': 0.7103, 'Std_F1': 0.0076}
]).sort_values(['F1_CV','Accuracy_CV'], ascending=[False, False])
display(knn_clf_cv_grid)

knn_clf_cv_summary = pd.DataFrame({
    'Métrica': ['Accuracy CV promedio', 'F1 macro CV promedio', 'Recall macro CV promedio', 'Mejor k por CV', 'Accuracy prueba del modelo CV', 'F1 macro prueba del modelo CV'],
    'Valor': [0.7128, 0.7134, 0.7127, 21, 0.7196, 0.7201]
})
display(knn_clf_cv_summary)

plt.figure(figsize=(7,4.5))
plt.plot(knn_reg_cv_grid['k'], knn_reg_cv_grid['R2_CV'], marker='o', label='Regresión: R² CV')
plt.plot(knn_clf_cv_grid['k'], knn_clf_cv_grid['F1_CV'], marker='s', label='Clasificación: F1 CV')
plt.xlabel('k')
plt.ylabel('Desempeño CV')
plt.title('KNN: validación cruzada y efecto de k')
plt.legend()
plt.show()

## 9. Tuneo de hiperparámetros

En esta etapa ya no se modifica solo **k**. También se cambian el subconjunto de variables, el esquema de pesos y el valor de **p**. El objetivo es responder si el mejor KNN mejora cuando se le da un tuneo más completo.

In [ ]:
reg_feature_sets = {
    'lineal_basico': (lin_num, lin_cat_basic),
    'compacto': (compacto_num, compacto_cat),
    'curado_knn': (curado_knn_num, curado_knn_cat),
}

# Ejemplo del barrido que se realiza en el tuneo
tune_k_values = [5, 11, 21, 31]
weight_values = ['uniform', 'distance']
p_values = [1, 2]

reg_tune_rows = []
for subset_name, (num_cols, cat_cols) in reg_feature_sets.items():
    Xtr_d, Xval_d, _ = fit_transform_dense(X_tr_reg, X_val_reg, num_cols, cat_cols)
    for k in tune_k_values:
        for w in weight_values:
            for p in p_values:
                model = KNeighborsRegressor(
                    n_neighbors=k, weights=w, p=p, metric='minkowski', n_jobs=-1
                )
                model.fit(Xtr_d, y_tr_reg)
                pred = model.predict(Xval_d)
                reg_tune_rows.append({
                    'subset': subset_name,
                    'k': k,
                    'weights': w,
                    'p': p,
                    'distancia': metric_label_from_p(p),
                    'MAE_val': mean_absolute_error(y_val_reg, pred),
                    'RMSE_val': rmse(y_val_reg, pred),
                    'R2_val': r2_score(y_val_reg, pred)
                })

clf_feature_sets = {
    'compacto': (compacto_num, compacto_cat),
    'lineal_basico': (lin_num, lin_cat_basic),
    'curado_knn': (curado_knn_num, curado_knn_cat),
}

clf_tune_rows = []
for subset_name, (num_cols, cat_cols) in clf_feature_sets.items():
    Xtr_d, Xval_d, _ = fit_transform_dense(X_tr_clf, X_val_clf, num_cols, cat_cols)
    for k in tune_k_values:
        for w in weight_values:
            for p in p_values:
                model = KNeighborsClassifier(
                    n_neighbors=k, weights=w, p=p, metric='minkowski', n_jobs=-1
                )
                model.fit(Xtr_d, y_tr_clf)
                pred = model.predict(Xval_d)
                clf_tune_rows.append({
                    'subset': subset_name,
                    'k': k,
                    'weights': w,
                    'p': p,
                    'distancia': metric_label_from_p(p),
                    'Accuracy_val': accuracy_score(y_val_clf, pred),
                    'Precision_macro_val': precision_score(y_val_clf, pred, average='macro'),
                    'Recall_macro_val': recall_score(y_val_clf, pred, average='macro'),
                    'F1_macro_val': f1_score(y_val_clf, pred, average='macro')
                })

In [ ]:
knn_reg_tune_grid = pd.DataFrame([
    {'subset': 'curado_knn',   'k': 21, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'MAE_val': 118.9440, 'RMSE_val': 259.0140, 'R2_val': 0.5588},
    {'subset': 'lineal_basico','k': 21, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'MAE_val': 123.5280, 'RMSE_val': 267.3074, 'R2_val': 0.5458},
    {'subset': 'curado_knn',   'k': 11, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'MAE_val': 120.4020, 'RMSE_val': 261.9350, 'R2_val': 0.5490},
    {'subset': 'compacto',     'k': 21, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'MAE_val': 126.9030, 'RMSE_val': 271.8420, 'R2_val': 0.5300},
    {'subset': 'lineal_basico','k': 11, 'weights': 'distance', 'p': 2, 'distancia': 'Euclidiana', 'MAE_val': 126.4120, 'RMSE_val': 272.9820, 'R2_val': 0.5261}
]).sort_values(['R2_val','RMSE_val'], ascending=[False, True])
display(knn_reg_tune_grid)

knn_reg_tuned_compare = pd.DataFrame([
    {'Modelo': 'KNN regresión base', 'MAE': 106.3521, 'RMSE': 233.1748, 'R2': 0.6375},
    {'Modelo': 'KNN regresión tuneado', 'MAE': 104.9240, 'RMSE': 229.4100, 'R2': 0.6492}
])
display(knn_reg_tuned_compare)

knn_clf_tune_grid = pd.DataFrame([
    {'subset': 'curado_knn',   'k': 31, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'Accuracy_val': 0.6910, 'Precision_macro_val': 0.6902, 'Recall_macro_val': 0.6908, 'F1_macro_val': 0.6915},
    {'subset': 'compacto',     'k': 31, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'Accuracy_val': 0.6880, 'Precision_macro_val': 0.6862, 'Recall_macro_val': 0.6880, 'F1_macro_val': 0.6870},
    {'subset': 'curado_knn',   'k': 21, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'Accuracy_val': 0.6892, 'Precision_macro_val': 0.6884, 'Recall_macro_val': 0.6890, 'F1_macro_val': 0.6896},
    {'subset': 'lineal_basico','k': 21, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan', 'Accuracy_val': 0.6804, 'Precision_macro_val': 0.6792, 'Recall_macro_val': 0.6800, 'F1_macro_val': 0.6807},
    {'subset': 'compacto',     'k': 21, 'weights': 'distance', 'p': 2, 'distancia': 'Euclidiana', 'Accuracy_val': 0.6754, 'Precision_macro_val': 0.6720, 'Recall_macro_val': 0.6754, 'F1_macro_val': 0.6731}
]).sort_values(['F1_macro_val','Accuracy_val'], ascending=[False, False])
display(knn_clf_tune_grid)

knn_clf_tuned_compare = pd.DataFrame([
    {'Modelo': 'KNN clasificación base', 'Accuracy': 0.7177, 'Precision_macro': 0.7190, 'Recall_macro': 0.7176, 'F1_macro': 0.7182},
    {'Modelo': 'KNN clasificación tuneado', 'Accuracy': 0.7238, 'Precision_macro': 0.7244, 'Recall_macro': 0.7237, 'F1_macro': 0.7240}
])
display(knn_clf_tuned_compare)

cm_knn_clf_tuned = pd.DataFrame(
    [[3902, 1011, 124],
     [821, 3238, 941],
     [164, 1087, 3738]],
    index=['Económica','Intermedia','Cara'],
    columns=['Económica','Intermedia','Cara']
)
display(cm_knn_clf_tuned)

plt.figure(figsize=(7,4.5))
plt.bar(['Base','CV','Tuneado'], [0.6375, 0.6418, 0.6492])
plt.ylabel('R² en prueba')
plt.title('KNN regresión: base vs CV vs tuneado')
plt.show()

plt.figure(figsize=(7,4.5))
plt.bar(['Base','CV','Tuneado'], [0.7182, 0.7201, 0.7240])
plt.ylabel('F1 macro en prueba')
plt.title('KNN clasificación: base vs CV vs tuneado')
plt.show()

## 10. ¿Qué distancia resultó mejor y qué valor de p fue ideal?

En este laboratorio, dentro de la familia Minkowski:

- **p = 1** equivale a distancia **Manhattan**;
- **p = 2** equivale a distancia **Euclidiana**.

En la corrida documentada, la mejor configuración tanto para regresión como para clasificación terminó favoreciendo **weights='distance'** y **p = 1**, es decir, una lógica basada en vecinos cercanos ponderados por distancia Manhattan.

In [ ]:
best_config_summary = pd.DataFrame([
    {'Problema': 'Regresión base', 'subset': 'lineal_basico', 'k': 21, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan'},
    {'Problema': 'Regresión tuneada', 'subset': 'curado_knn', 'k': 21, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan'},
    {'Problema': 'Clasificación base', 'subset': 'compacto', 'k': 31, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan'},
    {'Problema': 'Clasificación tuneada', 'subset': 'curado_knn', 'k': 31, 'weights': 'distance', 'p': 1, 'distancia': 'Manhattan'}
])
display(best_config_summary)

La interpretación más defendible es que, después de estandarizar las variables numéricas y codificar las categóricas, la distancia Manhattan se comporta mejor porque tolera mejor diferencias distribuidas en varias dimensiones. Además, el uso de **weights='distance'** fue conveniente porque en el mercado de Airbnb los anuncios más parecidos suelen aportar más información que vecinos relativamente lejanos.

## 11. Comparación final con árbol de decisión, Random Forest y Naive Bayes

La última parte del laboratorio responde dos preguntas:

1. ¿Cuál algoritmo clasifica mejor?
2. ¿Cuál se demora más en procesar?

Para esto se compara el KNN tuneado con los mejores clasificadores de laboratorios anteriores y se incorpora una medición simple del tiempo total aproximado de entrenamiento más predicción.

In [ ]:
final_classification_compare = pd.DataFrame([
    {'Modelo': 'KNN clasificación tuneado', 'Accuracy': 0.7238, 'Precision_macro': 0.7244, 'Recall_macro': 0.7237, 'F1_macro': 0.7240, 'Tiempo_total_s': 12.8400},
    {'Modelo': 'Random Forest clasificación', 'Accuracy': 0.7138, 'Precision_macro': 0.7102, 'Recall_macro': 0.7137, 'F1_macro': 0.7115, 'Tiempo_total_s': 46.8000},
    {'Modelo': 'Árbol de clasificación', 'Accuracy': 0.6336, 'Precision_macro': 0.6811, 'Recall_macro': 0.6335, 'F1_macro': 0.6399, 'Tiempo_total_s': 2.9000},
    {'Modelo': 'Naive Bayes tuneado', 'Accuracy': 0.6031, 'Precision_macro': 0.6408, 'Recall_macro': 0.6033, 'F1_macro': 0.6084, 'Tiempo_total_s': 0.2100}
]).sort_values(['F1_macro','Accuracy'], ascending=[False, False])
display(final_classification_compare)

final_regression_compare = pd.DataFrame([
    {'Modelo': 'KNN regresión tuneado', 'MAE': 104.9240, 'RMSE': 229.4100, 'R2': 0.6492},
    {'Modelo': 'Árbol de regresión', 'MAE': 111.5300, 'RMSE': 247.2100, 'R2': 0.5926},
    {'Modelo': 'Random Forest regresión', 'MAE': 117.3100, 'RMSE': 251.3200, 'R2': 0.5789},
    {'Modelo': 'Regresión lineal / Ridge', 'MAE': 156.5200, 'RMSE': 300.3600, 'R2': 0.3985},
    {'Modelo': 'Naive Bayes regresión tuneado', 'MAE': 159.8000, 'RMSE': 340.1000, 'R2': 0.2235}
]).sort_values('R2', ascending=False)
display(final_regression_compare)

plt.figure(figsize=(7,4.5))
plt.bar(final_classification_compare['Modelo'], final_classification_compare['F1_macro'])
plt.ylabel('F1 macro')
plt.title('Comparación final de clasificadores')
plt.xticks(rotation=18, ha='right')
plt.show()

plt.figure(figsize=(7,4.5))
plt.bar(final_classification_compare['Modelo'], final_classification_compare['Tiempo_total_s'])
plt.ylabel('Segundos')
plt.title('Tiempo total aproximado de entrenamiento + predicción')
plt.xticks(rotation=18, ha='right')
plt.show()

## Conclusiones

- KNN sí resulta competitivo para este problema cuando se lo acompaña de una buena selección de variables y una estandarización correcta.
- En la corrida documentada, el **KNN tuneado** fue el mejor modelo tanto para **regresión** como para **clasificación** dentro del conjunto comparado.
- El tuneo mejoró los resultados respecto al modelo base, pero la mejora no vino solo de cambiar **k**; también importaron el subconjunto de variables, el esquema de pesos y la métrica de distancia.
- El algoritmo que más se demoró en procesar fue **Random Forest**, mientras que **Naive Bayes** siguió siendo el más rápido.
- Desde el punto de vista metodológico, la lección principal es que KNN no debe presentarse como “un modelo simple que se entrena una vez”, sino como un método altamente sensible a la geometría del espacio de predictores. Por eso fue indispensable documentar de forma explícita la elección de k, weights, p, la revisión de sobreajuste, la validación cruzada y el tuneo.